In [1]:
import torch

# Check the PyTorch version.
print("PyTorch version:", torch.__version__)

# Check whether CUDA GPU is available.
print("CUDA available:", torch.cuda.is_available())

# Print the number of GPUs.
print("GPU count:", torch.cuda.device_count())

# Print the GPU name if CUDA is available.
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu130
CUDA available: True
GPU count: 1
Current GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [2]:
# Import the FastLanguageModel class from the Unsloth library
# This is Unsloth's optimized model loader for fast fine-tuning/inference
from unsloth import FastLanguageModel

# Load the model and its corresponding tokenizer in one line
model, tokenizer = FastLanguageModel.from_pretrained(
    # 1. The Hugging Face model name (Unsloth's pre-quantized Phi-3)
    model_name = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit',
    # 2. Maximum sequence length the model can handle (here set to 2048 tokens)
    max_seq_length = 2048,
    # 3. Data type (dtype) for weights; None means auto-detect based on GPU
    dtype = None,
    # 4. Enable 4-bit quantization (reduces VRAM usage drastically)
    # QLoRA, need 6 GB VRAM
    load_in_4bit = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0507 01:29:40.266000 26676 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti Laptop GPU. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:02<00:00, 144.64it/s]


In [3]:
# Import required libraries
import json
# Import the Dataset class from Hugging Face datasets library
from datasets import Dataset

# Open the JSON data file in read mode with UTF-8 encoding
with open("people_data.json", "r", encoding="utf-8") as f:
    # Load JSON data from the file into a Python list/dictionary
    data = json.load(f)

# Convert the loaded list/dict into a Hugging Face Dataset object
ds = Dataset.from_list(data)

In [4]:
def to_text(ex):
    """
    Convert a single example into a chat-formatted string using the tokenizer's chat template.
    Args:
        ex (dict): A single data example containing 'prompt' and 'response' keys
    Returns:
        dict: A new dictionary with a single 'text' key containing the formatted chat string
    """
    # Extract the response field from the example
    resp = ex["response"]
    # Check if the response is NOT a string (e.g. a dictionary/list)
    if not isinstance(resp, str):
        # Convert non-string responses to JSON string (preserve Chinese characters)
        resp = json.dumps(resp, ensure_ascii=False)

    # Create chat messages in the standard user-assistant format
    msgs = [
        {"role": "user", "content": ex["prompt"]},
        {"role": "assistant", "content": resp},
    ]

    # Return the formatted chat template as a single 'text' field
    return {
        "text": tokenizer.apply_chat_template(
            msgs,
            tokenize=False,          # Return plain text instead of token IDs
            add_generation_prompt=False  # Do NOT add the model's generation prompt suffix
        )
    }

In [5]:
# Apply the `to_text` function to every example in the dataset
# Remove all original columns, keeping only the new 'text' column
dataset = ds.map(to_text, remove_columns=ds.column_names)

Map: 100%|██████████| 20/20 [00:00<00:00, 1666.66 examples/s]


In [6]:
print(dataset[0])

{'text': '<|user|>\nAt midnight in a glass-walled data center, Mira, aged 29, works as a cybersecurity analyst. Outside of work, Mira designs puzzle rooms for friends.<|end|>\n<|assistant|>\n{"name": "Mira", "age": "29", "job": "cybersecurity analyst"}<|end|>\n<|endoftext|>'}


In [7]:
# Apply PEFT (LoRA) to the loaded model using Unsloth's optimized method
model = FastLanguageModel.get_peft_model(
    model,                                  # The base model loaded earlier (Phi-3 in this case)
    r = 64,                                 # Rank of the LoRA matrices (key hyperparameter for LoRA)
    target_modules=[                        # List of model layers to inject LoRA adapters into
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],                                      # These are the attention and MLP projection layers common in transformer models
    lora_alpha = 64 * 2,                    # Scaling factor for LoRA (typically set to 2x the rank)
    lora_dropout = 0,                       # LoRA dropout rate (0 means no dropout; increase for regularization)
    bias = 'none',                          # Bias parameter handling: 'none' means biases are frozen and not trained
    use_gradient_checkpointing = 'unsloth', # Enable Unsloth's custom gradient checkpointing (reduces GPU memory usage at the cost of more computation)
)

Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [8]:
# Import the supervised fine-tuning (SFT) trainer and config from the TRL library
from trl import SFTTrainer, SFTConfig
# Import Hugging Face's training arguments (base config for training)
from transformers import TrainingArguments

# Create the SFT trainer object
trainer = SFTTrainer(
    # The model we want to fine-tune (your LoRA-prepared Phi-3 model)
    model = model,
    # The dataset we prepared earlier (with the "text" column)
    train_dataset = dataset,
    # The tokenizer matching our model (from the previous step)
    tokenizer = tokenizer,
    # The name of the column in our dataset that contains the formatted text
    dataset_text_field = 'text',
    # Maximum token length the model will process (same as before: 2048)
    max_seq_length = 2048,
    # Training configuration settings
    args = SFTConfig(
        # Batch size per GPU: 2 sequences per GPU at once
        per_device_train_batch_size = 2,
        # Gradient accumulation: accumulate gradients over 4 steps before updating weights
        # Effective batch size = 2 * 4 = 8
        gradient_accumulation_steps = 4,
        # Warmup steps: slowly increase the learning rate from 0 over the first 10 steps
        warmup_steps = 10,
        # Max training steps: stop training after 60 steps (even if epochs are not done)
        max_steps = 60,
        # Log training progress every single step (for debugging)
        logging_steps = 1,
        # Folder to save checkpoints, logs, and the final model
        output_dir = "outputs",
        # Optimizer: use 8-bit AdamW (saves memory, works well with 4-bit models)
        optim = "adamw_8bit",
        # Number of epochs: loop through the full dataset 3 times
        # Training will stop early if max_steps (60) is reached first
        num_train_epochs = 3
    ),
)

# Start the supervised fine-tuning process
trainer.train()

Unsloth: Tokenizing ["text"]: 100%|██████████| 20/20 [00:00<00:00, 1666.29 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 20 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.552087
2,2.410735
3,2.537176
4,2.546244
5,2.314015
6,2.513004
7,2.357447
8,2.134603
9,2.063856
10,1.887811


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-60\tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.8289161799475551, metrics={'train_runtime': 64.3662, 'train_samples_per_second': 7.457, 'train_steps_per_second': 0.932, 'total_flos': 612279702650880.0, 'train_loss': 0.8289161799475551, 'epoch': 20.0})

In [9]:
import json
import torch
from unsloth import FastLanguageModel

# Switch model to inference mode for faster generation
FastLanguageModel.for_inference(model)

# ----------------------
# Load test dataset
# ----------------------
with open("people_test_data.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

# ----------------------
# CORRECT TEST FUNCTION (NO ERROR, NO WARNING)
# ----------------------
def test_single_example(example):
    """
    Test the fine-tuned model on one test case
    Args:
        example: dict with "prompt" and "expected_response"
    Returns:
        dict including prompt, expected answer, and model's output
    """
    # Get input prompt and expected correct answer
    prompt = example["prompt"]
    expected = example["expected_response"]

    # Build chat message format
    messages = [{"role": "user", "content": prompt}]

    # Tokenize input (WORKING VERSION FOR PHI-3)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate response (STABLE & NO ERROR)
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        use_cache=True,
        temperature=0.1,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    # Only keep the newly generated part (remove input prompt)
    input_length = inputs.shape[1]
    generated_text = tokenizer.batch_decode(
        outputs[:, input_length:],
        skip_special_tokens=True
    )[0].strip()

    # Return comparison result
    return {
        "prompt": prompt,
        "expected_response": expected,
        "model_response": generated_text
    }

In [10]:
# ----------------------
#  Run tests on all examples and print results
# ----------------------
print("=== Model Fine-Tuning Test Results ===\n")

for idx, example in enumerate(test_data, 1):
    result = test_single_example(example)

    print(f"Test Case {idx}:")
    print(f"Prompt: {result['prompt']}")
    print(f"Expected Response: {json.dumps(result['expected_response'], indent=2, ensure_ascii=False)}")
    print(f"Model's Response: {result['model_response']}")
    print("-" * 80 + "\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Model Fine-Tuning Test Results ===



D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be remove

Test Case 1:
Prompt: On a rainy afternoon in a comic book library, Leo, aged 30, works as a game writer. Leo designs mystery quests based on forgotten city rumors.
Expected Response: {
  "name": "Leo",
  "age": "30",
  "job": "game writer"
}
Model's Response: {"name": "Leo", "age": "30", "job": "game writer"}
--------------------------------------------------------------------------------



Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Test Case 33:
Prompt: At a bamboo bicycle workshop, Diego, age 38, earns a living as a bicycle frame builder. Diego tests every frame on steep city streets.
Expected Response: {
  "name": "Diego",
  "age": "38",
  "job": "bicycle frame builder"
}
Model's Response: {"name": "Diego", "age": "38", "job": "bicycle frame builder"}
--------------------------------------------------------------------------------



Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Case 34:
Prompt: In a children's science museum, the person is 45 and works as an exhibit designer. The person creates interactive displays about electricity.
Expected Response: {
  "name": "",
  "age": "45",
  "job": "exhibit designer"
}
Model's Response: {"name": "", "age": "45", "job": "exhibit designer"}
--------------------------------------------------------------------------------



Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Case 35:
Prompt: At a rooftop apiary, Celeste keeps honeybees for a living. Celeste paints each hive with a different constellation.
Expected Response: {
  "name": "Celeste",
  "age": "",
  "job": "beekeeper"
}
Model's Response: {"name": "Celeste", "age": "", "job": "honeybee keeper"}
--------------------------------------------------------------------------------

Test Case 36:
Prompt: Beside a frozen lake, Mikhail, aged 53, carves wooden animals and teaches children how to recognize animal tracks.
Expected Response: {
  "name": "Mikhail",
  "age": "53",
  "job": ""
}
Model's Response: {"name": "Mikhail", "age": "53", "job": ""}
--------------------------------------------------------------------------------



In [11]:
# ----------------------
# (Optional) Calculate simple accuracy for structured tasks
# ----------------------
def calculate_structured_accuracy(test_results):
    """
    Simple accuracy metric: count how many model responses exactly match the expected JSON
    """
    correct = 0
    total = len(test_results)
    for res in test_results:
        try:
            # Parse model's response as JSON and compare with expected
            model_json = json.loads(res["model_response"])
            if model_json == res["expected_response"]:
                correct += 1
        except json.JSONDecodeError:
            # If the model's response is not valid JSON, count as incorrect
            continue
    accuracy = correct / total * 100
    print(f"\nStructured Response Accuracy: {accuracy:.2f}% ({correct}/{total})")

# Collect all test results
all_results = [test_single_example(ex) for ex in test_data]

# Calculate and print accuracy
calculate_structured_accuracy(all_results)

Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


Structured Response Accuracy: 80.56% (29/36)
